# Notebook 10 – Imbalanced Classification

**Dataset:** Online Retail transactions (`data.csv`)

**Task:** Predict whether an order is from the **United Kingdom** or not, using `Quantity`, `UnitPrice`, `TotalPrice`.
This target is naturally **imbalanced** — most orders are from the UK (~90%).

## Setup: Load & Split Data

In [1]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
df = pd.read_csv('data.csv', encoding='latin1')
df = df.dropna(subset=['CustomerID'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].sample(3000, random_state=42)
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
X = df[['Quantity', 'UnitPrice', 'TotalPrice']]
y = (df['Country'] == 'United Kingdom').astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Class balance:\n", y.value_counts(normalize=True))

Class balance:
 Country
1    0.896667
0    0.103333
Name: proportion, dtype: float64


## 1. Class Imbalance
When one class makes up much more of the data than the other (here, ~90% UK vs ~10% non-UK). Many models naturally lean toward predicting the majority class, since that alone gets them a high score.

In [2]:
baseline_model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
preds = baseline_model.predict(X_test)
print("Predicted class distribution:", pd.Series(preds).value_counts().to_dict())

Predicted class distribution: {1: 600}


## 2. Why Accuracy Can Be Misleading
A model that **always predicts the majority class** (UK) can still score ~90% accuracy — without learning anything useful about the minority class at all.

In [4]:
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
print("Dummy 'always predict majority' accuracy:", accuracy_score(y_test, dummy.predict(X_test)))
print("(High accuracy, but the model never predicts the minority class at all!)")

Dummy 'always predict majority' accuracy: 0.8966666666666666
(High accuracy, but the model never predicts the minority class at all!)


## 3. Precision
Of everything predicted as the **minority class** (non-UK), what fraction was actually correct? Low precision means many false alarms.

In [5]:
print("Baseline model precision (for class 0, non-UK):",
      precision_score(y_test, preds, pos_label=0, zero_division=0))

Baseline model precision (for class 0, non-UK): 0.0


## 4. Recall
Of all the **actual** minority-class (non-UK) orders, what fraction did the model catch? On imbalanced data, recall for the minority class often reveals a model's real weakness.

In [6]:
print("Baseline model recall (for class 0, non-UK):",
      recall_score(y_test, preds, pos_label=0, zero_division=0))

Baseline model recall (for class 0, non-UK): 0.0


## 5. F1
The balance between precision and recall for the minority class — a much better single number than accuracy on imbalanced data.

In [7]:
print("Baseline model F1 (for class 0, non-UK):",
      f1_score(y_test, preds, pos_label=0, zero_division=0))

Baseline model F1 (for class 0, non-UK): 0.0


## 6. ROC-AUC
Measures how well the model ranks positive examples above negative ones, across **all** thresholds. Less sensitive to imbalance than accuracy, but can still look decent even with a weak minority-class model.

In [8]:
probs = baseline_model.predict_proba(X_test)[:, 1]
print("ROC-AUC:", roc_auc_score(y_test, probs))

ROC-AUC: 0.6165907183115482


## 7. PR-AUC
Precision-Recall AUC focuses specifically on the **positive (minority) class**'s precision/recall trade-off — often more informative than ROC-AUC on heavily imbalanced data.

In [9]:
pr_auc = average_precision_score(y_test, probs, pos_label=1)
pr_auc_minority = average_precision_score(1 - y_test, 1 - probs, pos_label=1)
print("PR-AUC (majority class):", pr_auc)
print("PR-AUC (minority class):", pr_auc_minority)

PR-AUC (majority class): 0.9392358492350437
PR-AUC (minority class): 0.15465260017495824


## 8. Class Weights
Tell the model to **penalize mistakes on the minority class more heavily** during training, without changing the data itself.

In [10]:
weighted_model = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X_train, y_train)
weighted_preds = weighted_model.predict(X_test)
print("Class-weighted — Recall (non-UK):", recall_score(y_test, weighted_preds, pos_label=0, zero_division=0))
print("Class-weighted — F1 (non-UK):     ", f1_score(y_test, weighted_preds, pos_label=0, zero_division=0))

Class-weighted — Recall (non-UK): 0.3064516129032258
Class-weighted — F1 (non-UK):      0.18357487922705315


## 9. Oversampling
**Duplicate** (or otherwise increase) examples from the minority class so the training set is more balanced.

In [11]:
from sklearn.utils import resample
train_df = X_train.copy(); train_df['y'] = y_train.values
majority = train_df[train_df.y == 1]
minority = train_df[train_df.y == 0]
minority_upsampled = resample(minority, replace=True, n_samples=len(majority), random_state=42)
oversampled = pd.concat([majority, minority_upsampled])
over_model = LogisticRegression(max_iter=1000).fit(oversampled.drop(columns='y'), oversampled['y'])
over_preds = over_model.predict(X_test)
print("Oversampling — Recall (non-UK):", recall_score(y_test, over_preds, pos_label=0, zero_division=0))
print("Oversampling — F1 (non-UK):     ", f1_score(y_test, over_preds, pos_label=0, zero_division=0))

Oversampling — Recall (non-UK): 0.3064516129032258
Oversampling — F1 (non-UK):      0.18269230769230768


## 10. Undersampling
**Remove** examples from the majority class so the training set is more balanced. Simpler than oversampling but throws away data.

In [14]:
majority_downsampled = resample(majority, replace=False, n_samples=len(minority), random_state=42)
undersampled = pd.concat([majority_downsampled, minority])
under_model = LogisticRegression(max_iter=1000).fit(undersampled.drop(columns='y'), undersampled['y'])
under_preds = under_model.predict(X_test)
print("Undersampling — Recall (non-UK):", recall_score(y_test, under_preds, pos_label=0, zero_division=0))
print("Undersampling — F1 (non-UK):     ", f1_score(y_test, under_preds, pos_label=0, zero_division=0))

Undersampling — Recall (non-UK): 0.3387096774193548
Undersampling — F1 (non-UK):      0.16666666666666666


## 11. SMOTE
Short for **Synthetic Minority Oversampling Technique**. Instead of simply duplicating minority examples, it creates **new, synthetic** minority examples by interpolating between existing ones — usually better than plain oversampling.

In [15]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X_train, y_train)
smote_model = LogisticRegression(max_iter=1000).fit(X_smote, y_smote)
smote_preds = smote_model.predict(X_test)
print("SMOTE — Recall (non-UK):", recall_score(y_test, smote_preds, pos_label=0, zero_division=0))
print("SMOTE — F1 (non-UK):     ", f1_score(y_test, smote_preds, pos_label=0, zero_division=0))

SMOTE — Recall (non-UK): 0.3225806451612903
SMOTE — F1 (non-UK):      0.18433179723502305


## 12. Threshold Adjustment
Instead of changing the data, **lower the decision threshold** for the minority class so the model predicts it more often — a quick fix that needs no retraining.

In [16]:
probs_minority = 1 - baseline_model.predict_proba(X_test)[:, 1]  
adjusted_preds = (probs_minority >= 0.11).astype(int)  
adjusted_preds = 1 - adjusted_preds  
print("Threshold-adjusted — Recall (non-UK):", recall_score(y_test, adjusted_preds, pos_label=0, zero_division=0))
print("Threshold-adjusted — F1 (non-UK):     ", f1_score(y_test, adjusted_preds, pos_label=0, zero_division=0))

Threshold-adjusted — Recall (non-UK): 0.12903225806451613
Threshold-adjusted — F1 (non-UK):      0.1509433962264151


## Comparing All Approaches

In [17]:
comparison = pd.DataFrame({
    'Approach': ['Baseline', 'Class Weights', 'Oversampling', 'Undersampling', 'SMOTE', 'Threshold Adjustment'],
    'Recall (non-UK)': [
        recall_score(y_test, preds, pos_label=0, zero_division=0),
        recall_score(y_test, weighted_preds, pos_label=0, zero_division=0),
        recall_score(y_test, over_preds, pos_label=0, zero_division=0),
        recall_score(y_test, under_preds, pos_label=0, zero_division=0),
        recall_score(y_test, smote_preds, pos_label=0, zero_division=0),
        recall_score(y_test, adjusted_preds, pos_label=0, zero_division=0),
    ],
    'F1 (non-UK)': [
        f1_score(y_test, preds, pos_label=0, zero_division=0),
        f1_score(y_test, weighted_preds, pos_label=0, zero_division=0),
        f1_score(y_test, over_preds, pos_label=0, zero_division=0),
        f1_score(y_test, under_preds, pos_label=0, zero_division=0),
        f1_score(y_test, smote_preds, pos_label=0, zero_division=0),
        f1_score(y_test, adjusted_preds, pos_label=0, zero_division=0),
    ],
    'Overall Accuracy': [
        accuracy_score(y_test, preds),
        accuracy_score(y_test, weighted_preds),
        accuracy_score(y_test, over_preds),
        accuracy_score(y_test, under_preds),
        accuracy_score(y_test, smote_preds),
        accuracy_score(y_test, adjusted_preds),
    ]
})
comparison

,Approach,Recall (non-UK),F1 (non-UK),Overall Accuracy
0,Baseline,0.000000,0.000000,0.896667
1,Class Weights,0.306452,0.183575,0.718333
2,Oversampling,0.306452,0.182692,0.716667
3,Undersampling,0.338710,0.166667,0.650000
4,SMOTE,0.322581,0.184332,0.705000
5,Threshold Adjustment,0.129032,0.150943,0.850000
